# Predicting steel yield strength - the Python behind Explore


## 1. Learning objectives

By the end of this notebook, you should be able to:

1. Explain the simplified Hall-Petch relationship and solid-solution strengthening.
2. Load and inspect a documented synthetic engineering dataset.
3. Separate training, validation and test data reproducibly.
4. Train and compare Linear Regression, a Decision Tree and a Random Forest.
5. Interpret R-squared and RMSE in the context of yield strength, measured in MPa.
6. Recognise overfitting and understand why test performance matters.
7. Check whether predictions follow the expected physical trends and remain within
   the demonstrated input ranges.
8. Understand why synthetic specimen scatter prevents a model from reproducing every
   generated value exactly.


## 2. The engineering problem

We want to predict the **yield strength** of a synthetic structural-steel sample from
six input variables:

- Carbon, manganese and silicon composition (wt%).
- Grain size (micrometres).
- Tempering temperature (degrees C).
- Test temperature (degrees C).

Yield strength is the stress at which a material begins to deform plastically. It is
an important material property, but predicting it accurately for real steel requires
validated experimental data and a much more complete understanding of metallurgy.

### Simplified physical relationship

The dataset is constructed using a Hall-Petch grain-size contribution and
approximately linear composition contributions:

\[
\sigma_y =
53.9 + \frac{550.2}{\sqrt{d}}
+ 500C + 32.3Mn + 83.2Si
- 0.20(T_{\mathrm{temper}}-400)
- 0.20(T_{\mathrm{test}}-20)
\]

Here, yield strength is in MPa, grain size \(d\) is in micrometres, composition variables
are in wt%, and temperatures are in degrees C.

The Hall-Petch term means that smaller grains increase the modelled yield strength, and
it is the main source of variation in this dataset. Because strength depends on
\(1/\sqrt{d}\), the effect is strongly **curved**: refining grains from 10 to 2 micrometres
adds far more strength than refining them from 50 to 10. Grain size is sampled from 2 to 50
micrometres on a logarithmic scale, so the curved fine-grain region is well represented.
The composition terms represent simplified strengthening contributions. The carbon
coefficient is a project-defined effective value: most carbon in these steels is present
as pearlite or carbides rather than in solution, so its effect is modest. The two
temperature corrections are project-defined empirical assumptions for this
demonstration, not universal material laws.

The generated target is then multiplied by a positive lognormal noise factor,
\(\exp(N(0, 0.04))\), to introduce synthetic specimen-to-specimen scatter.

This is an educational data-generation model, not a complete metallurgical model.
Real steel behaviour also depends on phase composition, heat-treatment history,
dislocation density, precipitates, texture, strain rate and many other factors.


## 3. Load the dataset

The notebook uses the bundled steel tensile-strength CSV generated by
`scripts/generate_steel_tensile_strength.py`.

When run from this repository, the notebook should load the local copy first.
A documented GitHub fallback allows the same notebook to run in Google Colab,
where the repository's local files are not available.

The dataset contains 500 synthetic rows, generated with random seed 42 and
lognormal noise sigma 0.04.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# One fixed seed, used everywhere, exactly as the browser Explore mode does.
SEED = 42
np.random.seed(SEED)

print("Library versions")
print("  numpy       ", np.__version__)
print("  pandas      ", pd.__version__)
import sklearn
print("  scikit-learn", sklearn.__version__)
print("Random seed:", SEED)

In [ ]:
CSV_NAME = "steel_tensile_strength_sample.csv"
RAW_URL = (
    "https://raw.githubusercontent.com/weiwangstfc/engineering-ml-studio/"
    "main/examples/" + CSV_NAME
)

# Try local locations first so the notebook works offline.
candidates = [
    os.path.join("..", "examples", CSV_NAME),  # running from notebooks/
    os.path.join("examples", CSV_NAME),         # running from the repo root
    CSV_NAME,                                   # running beside the CSV
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print(f"Loaded local dataset: {csv_path}")
else:
    df = pd.read_csv(RAW_URL)  # e.g. in Colab
    print(f"Loaded dataset from GitHub: {RAW_URL}")

print("Rows, columns:", df.shape)
df.head()

## 4. Choose the inputs and the target

We name the six inputs and the single target explicitly. Keeping units and
measurement conventions clear in the column names helps prevent mistakes
when preparing engineering data.

In [ ]:
FEATURES = [
    "carbon_wt_pct",
    "manganese_wt_pct",
    "silicon_wt_pct",
    "grain_size_um",
    "tempering_temperature_c",
    "test_temperature_c",
]
TARGET = "yield_strength_mpa"
TARGET_UNIT = "MPa"

X = df[FEATURES].to_numpy()
y = df[TARGET].to_numpy()

print("Inputs (features):")
for f in FEATURES:
    print("  -", f)
print("Target:", TARGET, f"({TARGET_UNIT})")
print("Number of rows:", len(y))

## 5. Inspect the data before modelling

Always *look* at the data first. The summary statistics show the range of each input and of
yield strength. The two scatter plots show the most important relationship in this dataset.
On the left, yield strength against grain size is clearly **curved**. On the right, the same
data plotted against \(1/\sqrt{d}\) falls roughly on a straight line: this is the Hall-Petch
plot that metallurgists use. The remaining spread comes from the other inputs and the
specimen scatter. These are relationships in our synthetic dataset, not universal guarantees
for every real steel grade.

In [ ]:
df[FEATURES + [TARGET]].describe().T

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(
    df["grain_size_um"], df[TARGET],
    s=12, alpha=0.5, color="#125f76"
)
axes[0].set_xlabel("Grain size (micrometres)")
axes[0].set_ylabel(f"Yield strength ({TARGET_UNIT})")
axes[0].set_title("Strength against grain size is curved")

axes[1].scatter(
    df["grain_size_um"] ** -0.5, df[TARGET],
    s=12, alpha=0.5, color="#a52b2b"
)
axes[1].set_xlabel("1 / sqrt(grain size)  (micrometres^-0.5)")
axes[1].set_ylabel(f"Yield strength ({TARGET_UNIT})")
axes[1].set_title("Hall-Petch plot: roughly a straight line")

fig.tight_layout()
plt.show()


## 6. Split into training and test sets (reproducibly)

We hold back data the model never sees during training, so we can judge it honestly. Explore uses a
**70 % / 15 % / 15 %** split (training / validation / test) with a fixed seed. We reproduce the same
ratio and the same seed here.

**Why a fixed seed?** `random_state=42` makes the split repeatable: run the notebook again, or on
another machine, and you get the same partition - so results are comparable.

> **Note.** These three models are *not* tuned, so they do not use the validation set. It is created
> here only to mirror Explore's split; tuning on it is an advanced extension (Section 15). We train
> on the training set and report on the **test** set.

In [ ]:
# 70 / 15 / 15 : first hold out 30% (temp), then halve it into validation and test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"Training rows:   {len(y_train)}")
print(f"Validation rows: {len(y_val)}  (unused by these fixed models)")
print(f"Test rows:       {len(y_test)}")

## 7. The simple trend first: Linear Regression

**Linear Regression** learns a *weighted relationship*: each input is multiplied by a number and
the results are added up. It is easy to interpret, but it cannot directly represent curved
relationships such as the Hall-Petch relationship between grain size and yield strength.

We put a `StandardScaler` in front of the linear model. Scaling puts the inputs on a comparable
footing; it does not change the quality of a plain linear fit, but it is good practice and matches
what Explore does internally.


In [ ]:
def rmse(y_true, y_pred):
    'Root-mean-square error, in the same units as the target (MPa).'
    return mean_squared_error(y_true, y_pred) ** 0.5

linear = make_pipeline(StandardScaler(), LinearRegression())
linear.fit(X_train, y_train)

lin_pred_train = linear.predict(X_train)
lin_pred_test = linear.predict(X_test)

print("Linear Regression")
print(f"  Train : R2 = {r2_score(y_train, lin_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, lin_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, lin_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, lin_pred_test):6.2f} {TARGET_UNIT}")

## 8. A more flexible relationship - Random Forest

A **Random Forest** combines many decision trees and averages them. It handles curved,
non-linear relationships well and is more stable than any single tree. The price is that it is
**less transparent** than a straight-line model, and it **extrapolates poorly** beyond the range of
the training data (more on that in Section 13).

We use scikit-learn's default forest settings with our fixed seed.

> **A note on settings.** This notebook uses scikit-learn's default Random Forest settings
> with a fixed random seed. Its results may differ from the browser implementation because
> the model configurations and training environments are not necessarily identical. Focus on
> the comparison between training and test performance rather than expecting identical scores.

In [ ]:
forest = RandomForestRegressor(random_state=SEED)  # scikit-learn defaults
forest.fit(X_train, y_train)

rf_pred_train = forest.predict(X_train)
rf_pred_test = forest.predict(X_test)

print("Random Forest")
print(f"  Train : R2 = {r2_score(y_train, rf_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, rf_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, rf_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, rf_pred_test):6.2f} {TARGET_UNIT}")

## 9. Seeing overfitting - a single Decision Tree

A **Decision Tree** splits the inputs into regions with a series of yes/no questions and predicts a
value for each region. It can capture non-linear behaviour, but it can overfit the training data.

Comparing its training and test performance helps us identify whether it is learning patterns
that generalise to unseen observations.

In [ ]:
tree = DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=SEED)
tree.fit(X_train, y_train)

dt_pred_train = tree.predict(X_train)
dt_pred_test = tree.predict(X_test)

print("Decision Tree (max_depth=8, min_samples_leaf=5)")
print(f"  Train : R2 = {r2_score(y_train, dt_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, dt_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, dt_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, dt_pred_test):6.2f} {TARGET_UNIT}")
print()
gap = rmse(y_test, dt_pred_test) - rmse(y_train, dt_pred_train)
print(f"Test RMSE is {gap:.2f} {TARGET_UNIT} larger than training RMSE "
      f"-> a sign of overfitting.")

## 10. Compare the models - RMSE and R²

Now we put the three models side by side on the **test** set (the honest measure) and the training
set (to reveal overfitting).

- **R²** is the fraction of the variation in yield strength the model explains: 1 is perfect, 0 is no
  better than always predicting the average.
- **RMSE** is the typical size of the prediction error, in **MPa** - smaller is better.

In [ ]:
def scores(name, model_pred_train, model_pred_test):
    return {
        "Model": name,
        "Train R2": round(r2_score(y_train, model_pred_train), 3),
        "Test R2": round(r2_score(y_test, model_pred_test), 3),
        f"Train RMSE ({TARGET_UNIT})": round(rmse(y_train, model_pred_train), 2),
        f"Test RMSE ({TARGET_UNIT})": round(rmse(y_test, model_pred_test), 2),
    }

results = pd.DataFrame([
    scores("Linear Regression", lin_pred_train, lin_pred_test),
    scores("Decision Tree", dt_pred_train, dt_pred_test),
    scores("Random Forest", rf_pred_train, rf_pred_test),
])
results

**Reading the table.** Linear Regression does *acceptably*: it captures the general trend,
but it cannot follow the curved Hall-Petch relationship, so its test error is the largest of the
three. Both tree-based models follow the curve and do clearly better on the test set, with the
Random Forest best.

Look at the training columns as well. The Decision Tree and the Random Forest fit the training
data much more closely than they fit the test data. The single tree has a large train-test gap,
which is a sign of overfitting; averaging many trees in the forest reduces it.

Even the best model does not reach an R-squared of 1. The dataset includes about 4% synthetic
specimen-to-specimen scatter, which no model can predict: it sets a noise floor. Real tensile
tests scatter too. These results are a deterministic baseline for this synthetic dataset, not a
guaranteed ranking for real steel data or other datasets.

## 11. Actual vs predicted (the best model)

Each point is one **test** row: the true yield strength on the horizontal axis, the model's
prediction on the vertical axis. Points on the dashed diagonal are perfect predictions; a tight
cloud along the line means accurate predictions.

In [ ]:
# Pick the model with the highest test R-squared rather than assuming one.
test_predictions = {
    "Linear Regression": lin_pred_test,
    "Decision Tree": dt_pred_test,
    "Random Forest": rf_pred_test,
}
best_name = max(test_predictions, key=lambda name: r2_score(y_test, test_predictions[name]))
best_pred = test_predictions[best_name]

lo = float(min(y_test.min(), best_pred.min()))
hi = float(max(y_test.max(), best_pred.max()))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_pred, s=28, alpha=0.7, color="#125f76", label="Test rows")
ax.plot([lo, hi], [lo, hi], "--", color="#a52b2b", label="Perfect prediction")
ax.set_xlabel(f"Actual yield strength ({TARGET_UNIT})")
ax.set_ylabel(f"Predicted yield strength ({TARGET_UNIT})")
ax.set_title(f"{best_name}: actual vs predicted (test data)")
ax.legend()
fig.tight_layout()
plt.show()

## 12. Overfitting, and why the test score is the one that counts

A low **training** error on its own is never enough - a model can memorise training rows. What
matters is performance on data it has **not** seen. The chart below compares training and test RMSE
for all three models. A tall jump from training to test (as with the single tree) is overfitting.

In [ ]:
names = ["Linear Regression", "Decision Tree", "Random Forest"]
train_rmse = [
    rmse(y_train, lin_pred_train),
    rmse(y_train, dt_pred_train),
    rmse(y_train, rf_pred_train),
]
test_rmse = [
    rmse(y_test, lin_pred_test),
    rmse(y_test, dt_pred_test),
    rmse(y_test, rf_pred_test),
]

x = np.arange(len(names))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w / 2, train_rmse, w, label="Training RMSE", color="#8fc7d4")
ax.bar(x + w / 2, test_rmse, w, label="Test RMSE", color="#125f76")
ax.set_xticks(x, names)
ax.set_ylabel(f"RMSE ({TARGET_UNIT})")
ax.set_title("Training vs test error (mind the gap)")
ax.legend()
fig.tight_layout()
plt.show()

## 13. Interpret it like an engineer - trends, plausibility, and staying in range

Good test scores are not the whole story. Before trusting a model, we check that it behaves
consistently with the relationships built into the synthetic dataset, that its predictions are
physically plausible, and that we are using it inside the range of data it learned from.

**(a) Does the model reproduce the expected trends?** We hold every input at its median and vary
one input at a time from a low value to a high value. We then compare the direction of the
predicted change with the relationship specified by the synthetic generator.

In [ ]:
medians = df[FEATURES].median()

def trend_effect(feature, model):
    'Predict at the 10th and 90th percentile of one feature, others at median.'
    low = df[feature].quantile(0.10)
    high = df[feature].quantile(0.90)
    row_low = medians.copy()
    row_low[feature] = low
    row_high = medians.copy()
    row_high[feature] = high
    predictions = model.predict(np.array([
        row_low[FEATURES].to_numpy(),
        row_high[FEATURES].to_numpy()
    ]))
    return predictions[1] - predictions[0]

expected = {
    "carbon_wt_pct": "increase",
    "manganese_wt_pct": "increase",
    "silicon_wt_pct": "increase",
    "grain_size_um": "decrease",
    "tempering_temperature_c": "decrease",
    "test_temperature_c": "decrease",
}

print("Trend checks for Linear Regression:")
print("(Other inputs held at their median values.)\n")

for feature, want in expected.items():
    delta = trend_effect(feature, linear)
    got = "increase" if delta > 0 else ("decrease" if delta < 0 else "no change")
    ok = "OK  " if got == want else "WARN"
    print(
        f"  [{ok}] as {feature:28s} rises, yield strength "
        f"{got:8s} by approximately {abs(delta):6.2f} {TARGET_UNIT} "
        f"(expected {want})"
    )

**(b) Are the predictions physically plausible?** Yield strength should be positive for the
steel specimens represented by this dataset. A negative prediction would be physically
implausible.

In [ ]:
negative_predictions = int((lin_pred_test < 0).sum())

if negative_predictions == 0:
    print(
        f"None of the {len(lin_pred_test)} test predictions are negative. "
        "All are positive."
    )
else:
    print(
        f"WARNING: {negative_predictions} of {len(lin_pred_test)} test predictions "
        "are negative and physically implausible."
    )

**(c) Are we using the model inside its demonstrated range?** A model learns from the input
ranges it was shown. Predictions outside those ranges are extrapolations and may be unreliable.
The check below flags an example with a grain size beyond the synthetic training range.

It uses the linear model on purpose. A straight line fitted mostly to the steep, fine-grain part
of the Hall-Petch curve keeps falling at the same rate as grain size grows, whereas the real curve
flattens out. Far outside the training range the linear prediction can become physically
impossible, even negative.

In [ ]:
train_df = pd.DataFrame(X_train, columns=FEATURES)
lows, highs = train_df.min(), train_df.max()

print("Demonstrated training input ranges:")
for feature in FEATURES:
    print(f"  {feature:28s}: {lows[feature]:.4g} .. {highs[feature]:.4g}")

def in_domain(row):
    'True if every input is within its demonstrated training range.'
    return all(lows[feature] <= row[feature] <= highs[feature]
               for feature in FEATURES)

example = medians.to_dict()
example["grain_size_um"] = 100.0

prediction = linear.predict(
    np.array([[example[feature] for feature in FEATURES]])
)[0]

print(f"\nExample prediction: {prediction:.2f} {TARGET_UNIT}")
print(
    "Inside demonstrated range?",
    in_domain(example),
    "-> treat out-of-range predictions with great caution."
)

> **Engineering takeaway.** These checks examine whether the model behaves consistently with
> the assumptions used to generate this synthetic dataset. They do not validate a real steel
> material model. The data are not experimental or suitable for material selection, structural
> design, safety-critical qualification, certification, or engineering decisions. Real steel
> behaviour also depends on factors omitted from this simplified model, including phase
> composition, heat treatment, dislocation density, precipitates, texture and strain rate.
> Machine learning supports engineering investigation; it does not replace engineering judgement.

## 14. Your turn - short exercises

Try these by editing the cells above or adding new cells. They build intuition about the
steel dataset and the behaviour of different models:

1. **Change the seed.** Set `SEED = 7` at the top and re-run. Do the metrics change?
   Does the ranking of the models change?
2. **Drop an input.** Remove `"silicon_wt_pct"` from `FEATURES` and re-run. Which model
   is most affected?
3. **Change the forest size.** Try `RandomForestRegressor(n_estimators=20, random_state=SEED)`
   and `n_estimators=300`. Does adding trees always improve the test score?
4. **Make the tree overfit harder.** Set the Decision Tree to `max_depth=None`. What happens
   to the training and test performance gap?
5. **Explore the Hall-Petch relationship.** Change `grain_size_um` in a prediction row.
   What happens to the predicted yield strength as grain size increases?
6. **Predict your own specimen.** Build an input row inside all six demonstrated ranges.
   Does the predicted yield strength look consistent with the trends?

In [ ]:
# Scratch space for the exercises. Edit and run freely.
# Example for exercise 6: an in-range synthetic steel specimen.

my_row = {
    "carbon_wt_pct": 0.08,
    "manganese_wt_pct": 1.00,
    "silicon_wt_pct": 0.25,
    "grain_size_um": 15.0,
    "tempering_temperature_c": 400.0,
    "test_temperature_c": 20.0,
}

my_pred = linear.predict(
    np.array([[my_row[feature] for feature in FEATURES]])
)[0]

print(f"Predicted yield strength: {my_pred:.2f} {TARGET_UNIT}")
print("Inside demonstrated range?", in_domain(my_row))

## 15. Where to go next (optional, advanced)

These are optional extensions once the main workflow is understood:

- **Feature importance:** inspect which inputs the Random Forest relied on most. This
  describes the fitted model, not the causal importance of real metallurgical factors.
- **Cross-validation:** use `sklearn.model_selection.cross_val_score` to estimate
  performance across several train and test partitions.
- **Gradient boosting:** try `sklearn.ensemble.GradientBoostingRegressor` or
  `HistGradientBoostingRegressor` as alternative models.
- **Prediction intervals:** investigate ways to represent uncertainty rather than
  reporting only a single predicted value.
- **Your own non-critical data:** try another dataset while retaining the cautions about
  synthetic data, validation and predictions outside the training range.

Remember the framing throughout: this is a learning tool using synthetic data, not a
validated material model or engineering design tool.

In [ ]:
# Feature importance from the Random Forest.
# These values describe this fitted model, not causal effects in real steel.
importances = pd.Series(
    forest.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("Random Forest feature importance:")
for name, val in importances.items():
    print(f"  {name:28s} {val:.3f}")

## 16. Optional advanced extension - a neural network

> **Optional and advanced.** Everything required from this notebook is already covered
> above. This section introduces a neural network using `MLPRegressor`, the same kind of
> model offered as the advanced option in browser Explore mode. It demonstrates the code
> and allows us to compare its results with the three standard models.

A **neural network**, here a multi-layer perceptron, learns by combining layers of simple
units. Each unit takes a weighted sum of its inputs, applies a non-linear activation
function (we use `relu`) and passes the result to the next layer. Stacking layers allows
the network to represent non-linear relationships such as the Hall-Petch contribution.

A few things matter for neural networks:

- **Scaling is essential.** The steel inputs have very different numerical ranges and
  units. We use `StandardScaler` in the same pipeline as the network. The scaler is
  fitted on training data only and then applied consistently.
- **Architecture:** `hidden_layer_sizes=(32, 16)` means two hidden layers, with 32 and
  then 16 neurons. More or larger layers add capacity, but also complexity and a risk
  of overfitting.
- **Training length and learning rate:** training is iterative. `max_iter` sets the
  iteration limit, while `learning_rate_init` controls the initial step size.
- **Early stopping and regularisation:** `early_stopping=True` reserves part of the
  training data for validation and stops when the validation score stops improving.
  `alpha` adds an L2 penalty to the weights.
- **Reproducibility:** `random_state=SEED` fixes the random initialisation and internal
  shuffling for repeatable results in the same environment.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor

# StandardScaler + MLPRegressor in one pipeline, so scaling is learned on the training data only.
# hidden_layer_sizes=(32, 16) mirrors Explore's "Medium network" preset.
nn = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16),   # two hidden layers: 32 then 16 neurons
        activation="relu",
        solver="adam",
        alpha=1e-3,                     # L2 regularisation (weight penalty)
        learning_rate_init=0.01,
        max_iter=2000,                  # generous epoch cap; early stopping usually ends sooner
        early_stopping=True,            # hold out 10% of training data to watch for overfitting
        n_iter_no_change=25,            # patience before stopping
        random_state=SEED,
    ),
)

# A neural network may warn if it hits max_iter without fully converging. We do NOT hide that:
# we capture it and report it honestly, because it tells you something about the training.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    nn.fit(X_train, y_train)

mlp = nn.named_steps["mlpregressor"]
nn_pred_train = nn.predict(X_train)
nn_pred_test = nn.predict(X_test)

converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
print("Neural network (MLPRegressor, hidden layers = (32, 16))")
print(f"  Train : R2 = {r2_score(y_train, nn_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, nn_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, nn_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, nn_pred_test):6.2f} {TARGET_UNIT}")
print(f"  Trained for {mlp.n_iter_} iterations "
      f"({'converged / stopped early' if converged else 'hit max_iter - see note below'}).")

### The training (loss) curve

Because a neural network trains **iteratively**, we can watch its **loss** fall iteration by iteration.
`MLPRegressor` records this in `loss_curve_`. A curve that falls and then flattens has effectively
converged; if it were still dropping steeply at the end, a larger `max_iter` might help. This is a
diagnostic - the numbers that matter for the engineering question are still the **test** RMSE and R².

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(
    range(1, len(mlp.loss_curve_) + 1),
    mlp.loss_curve_,
    color="#125f76"
)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Training loss")
axes[0].set_title("Neural network training curve")

lo = float(min(y_test.min(), nn_pred_test.min()))
hi = float(max(y_test.max(), nn_pred_test.max()))

axes[1].scatter(
    y_test, nn_pred_test,
    s=28, alpha=0.7, color="#125f76", label="Test rows"
)
axes[1].plot(
    [lo, hi], [lo, hi],
    "--", color="#a52b2b", label="Perfect prediction"
)
axes[1].set_xlabel(f"Actual yield strength ({TARGET_UNIT})")
axes[1].set_ylabel(f"Predicted yield strength ({TARGET_UNIT})")
axes[1].set_title("Neural network: actual vs predicted (test)")
axes[1].legend()

fig.tight_layout()
plt.show()

# Add the neural network to the comparison table.
results_with_nn = pd.concat([
    results,
    pd.DataFrame([scores("Neural Network", nn_pred_train, nn_pred_test)]),
], ignore_index=True)

results_with_nn

### Comparing early stopping with longer training

The early-stopping network provides a useful starting point, but its result depends on
when training stops.

The cell above reports how many iterations the early-stopping network ran and its test
scores. Below, we train the same architecture without early stopping and compare the two.

This is an experiment about training length. It does not mean that disabling early
stopping is always preferable.

In [ ]:
# Compare the same architecture with early stopping disabled.
nn_converged = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=0.01,
        max_iter=2000,
        early_stopping=False,
        random_state=SEED,
    ),
)

with warnings.catch_warnings(record=True) as caught_converged:
    warnings.simplefilter("always", ConvergenceWarning)
    nn_converged.fit(X_train, y_train)

conv_pred_train = nn_converged.predict(X_train)
conv_pred_test = nn_converged.predict(X_test)
conv_mlp = nn_converged.named_steps["mlpregressor"]

conv_converged = not any(
    issubclass(w.category, ConvergenceWarning)
    for w in caught_converged
)

print("Same architecture, early_stopping=False:")
print(f"  Iterations: {conv_mlp.n_iter_}")
print(f"  Train R2:   {r2_score(y_train, conv_pred_train):.3f}")
print(f"  Test R2:    {r2_score(y_test, conv_pred_test):.3f}")
print(f"  Test RMSE:  {rmse(y_test, conv_pred_test):.2f} {TARGET_UNIT}")
print(f"  Convergence warning: {not conv_converged}")

print("\nEarly-stopping comparison:")
print(f"  Iterations: {mlp.n_iter_}")
print(f"  Test R2:    {r2_score(y_test, nn_pred_test):.3f}")
print(f"  Test RMSE:  {rmse(y_test, nn_pred_test):.2f} {TARGET_UNIT}")

*Engineering ML Studio is a redevelopment of **Local Regression Studio** by **Yu Duan**
(MIT licence). New contributions in this project are copyright UKRI and released under the Apache-2.0
licence; they do not alter the original work's copyright or attribution.

This notebook and its dataset are a **synthetic teaching demonstration**. They are not
experimental, validated, safety-grade or design-quality data. They must not be used for
material selection, structural design, safety-critical qualification, certification or
engineering decisions.